# Annotated GIF exporter
Loads the configured experiment, stacks the first 100 frames, superimposes a 100 µm bar and clock, and writes a GIF at the true frame rate.


In [3]:
# Choose GIF source: 'processed', 'streamlines', or 'raw'.
FRAME_SOURCE = 'processed'  # options: 'processed', 'streamlines', 'raw'
FRAME_LIMIT = 100


In [4]:
import json
from pathlib import Path
from typing import Optional
import re

import cv2
from PIL import Image
import yaml

CONFIG_PATH = Path('../configs/opencv_tracker_v3.yaml').resolve()
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) or {}
input_cfg = config.get('input', {})
config_dir = CONFIG_PATH.parent

def resolve_path(path_like, base_dir: Path) -> Path:
    candidate = Path(path_like)
    return candidate if candidate.is_absolute() else (base_dir / candidate).resolve()

def natural_sort(paths):
    def key(path: Path):
        tokens = re.split(r'(\d+)', path.name)
        return [int(tok) if tok.isdigit() else tok.lower() for tok in tokens]
    return sorted(paths, key=key)

BASE_DIR = resolve_path(input_cfg['base_dir'], config_dir)
RAW_ROOT = resolve_path(input_cfg.get('raw_root', BASE_DIR.parent), config_dir)
PROCESSED_ROOT = resolve_path(
    config.get('output', {}).get('processed_root', RAW_ROOT.parent / 'processed'),
    config_dir,
)
try:
    RELATIVE = BASE_DIR.relative_to(RAW_ROOT)
except ValueError:
    RELATIVE = Path(BASE_DIR.name)
PROCESSED_EXPERIMENT = PROCESSED_ROOT / RELATIVE
CROPS_SUBDIR = config.get('output', {}).get('crops_subdir', 'cropped') or 'cropped'
SUMMARY_PATH = PROCESSED_EXPERIMENT / config.get('output', {}).get('summary_json', 'frame_rate_summary.json')
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'Frame-rate summary missing: {SUMMARY_PATH}')
SUMMARY = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
FOLDER_SUMMARIES = SUMMARY.get('folder_summaries', [])
if not FOLDER_SUMMARIES:
    raise RuntimeError('No folder summaries were produced yet.')
FOLDER_INFO = FOLDER_SUMMARIES[0]
SUBFOLDER = FOLDER_INFO['subfolder']
FRAME_RATE = float(FOLDER_INFO.get('frame_rate_hz', 15))
METADATA_PATH = PROCESSED_EXPERIMENT / 'metadata.json'
if not METADATA_PATH.exists():
    METADATA_PATH = PROCESSED_EXPERIMENT / 'metadata.csv'
METADATA_RECORD = {}
if METADATA_PATH.exists():
    if METADATA_PATH.suffix.lower() == '.json':
        METADATA_RECORD = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
    else:
        import csv

        with METADATA_PATH.open(encoding='utf-8') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                METADATA_RECORD = row
                break
PIXEL_PER_UM = float(METADATA_RECORD.get('pixelperum', 1.36)) if METADATA_RECORD else 1.36
CROPPED_DIR = PROCESSED_EXPERIMENT / CROPS_SUBDIR / f"{SUBFOLDER}_cropped"
RAW_DIR = BASE_DIR / SUBFOLDER
STREAMLINES_DIR = PROCESSED_EXPERIMENT / 'streamlines' / SUBFOLDER
STREAMLINES_DIR_ALT = PROCESSED_EXPERIMENT / 'streamlines'
SOURCE_DIRS = []
if FRAME_SOURCE == 'processed':
    SOURCE_DIRS.extend([CROPPED_DIR, STREAMLINES_DIR, STREAMLINES_DIR_ALT])
elif FRAME_SOURCE == 'streamlines':
    SOURCE_DIRS.extend([STREAMLINES_DIR, STREAMLINES_DIR_ALT, CROPPED_DIR])
elif FRAME_SOURCE == 'raw':
    SOURCE_DIRS.append(RAW_DIR)
else:
    raise ValueError(f"Unexpected FRAME_SOURCE '{FRAME_SOURCE}'")
SOURCE_DIRS.append(RAW_DIR)
SEARCH_DIR: Optional[Path] = None
for candidate in SOURCE_DIRS:
    if candidate and candidate.exists():
        SEARCH_DIR = candidate
        break
if SEARCH_DIR is None:
    raise FileNotFoundError('Unable to locate any frame folder for the selected source.')
if SEARCH_DIR == STREAMLINES_DIR and FRAME_SOURCE != 'streamlines':
    print('Using streamlines folder from processed experiment')
elif SEARCH_DIR == STREAMLINES_DIR_ALT and FRAME_SOURCE != 'streamlines':
    print('Using streamlines root folder from processed experiment')

FRAME_EXTS = ('.bmp', '.tif', '.png')
IMAGE_PATHS = natural_sort(
    [p for p in SEARCH_DIR.iterdir() if not p.name.startswith('.') and p.suffix.lower() in FRAME_EXTS]
)
if not IMAGE_PATHS:
    raise RuntimeError('No frames were found to build the GIF.')
IMAGE_PATHS = IMAGE_PATHS[:FRAME_LIMIT]

def rasterize_text(frame, text, position):
    cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

def overlay(frame):
    img = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR) if len(frame.shape) == 2 else frame.copy()
    height, width = img.shape[:2]
    margin = 20
    bar_len = max(5, int(round(100 / PIXEL_PER_UM)))
    start_x = width - margin - bar_len
    y = height - margin
    cv2.rectangle(img, (start_x, y - 4), (start_x + bar_len, y + 4), (255, 255, 255), -1)
    return img

frames = []
for idx, path in enumerate(IMAGE_PATHS):
    frame = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if frame is None:
        continue
    annotated = overlay(frame)
    elapsed = idx / FRAME_RATE
    rasterize_text(annotated, f'Time {elapsed:.2f} s', (20, 40))
    frames.append(Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))

if not frames:
    raise RuntimeError('No frames could be annotated.')

GIF_PATH = PROCESSED_EXPERIMENT / f'scale_bar_{SUBFOLDER}_{FRAME_SOURCE}.gif'
frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=frames[1:],
    duration=int(round(1000.0 / FRAME_RATE)),
    loop=0,
)
print('Saved GIF:', GIF_PATH)


RuntimeError: No frames could be annotated.

In [5]:
IMAGE_PATHS

[PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._106_4127_crop.tif'),
 PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._107_4128_crop.tif'),
 PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._108_4129_crop.tif'),
 PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._109_4130_crop.tif'),
 PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._110_4131_crop.tif'),
 PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._111_4132_crop.tif'),
 PosixPath('/Volumes/Extreme SSD/deep-sea-particles-gm/processed/pyrite_2024-11-14 20-00-15.825503_good/cropped/0_cropped/._112_41